# Building a cortical surface mesh from a GM segmentation volume

Produces `corticalSurface{age}mo.mat` (read by every mesh-visualisation notebook in this project) via marching cubes + Laplacian smoothing on a grey-matter segmentation volume, plus left/right hemisphere-split versions for NeuroDOT/MATLAB use.

In [ ]:
import nibabel as nib
from skimage import measure
import numpy as np
import trimesh
from trimesh.smoothing import filter_laplacian
import scipy.io as sio

from pathlib import Path

timePoint = '12'  # '01', '06' or '12'

# ---- USER SETTINGS ----
# Root of the drive/mount holding the MRI/mesh data (the same `driveName` used
# throughout the imageReconstruction MATLAB repo). Edit for your machine -
# pathlib.Path handles the rest cross-platform, no OS-specific branching needed.
dataDriveRoot = Path('/Volumes/YourDriveName')  # <-- edit: external drive mount point, or a local/mapped folder

gmVolPath = dataDriveRoot / 'mri' / 'registered' / 'UNC_to_NeuroDev' / 'No Mask' / f'{timePoint}mo' / f'{timePoint}mo_GM_Reg_Head_MeshVol.nii.gz'
meshesDir = dataDriveRoot / 'mri' / 'meshes'
meshesDir.mkdir(parents=True, exist_ok=True)


## Build the mesh (once)

Runs marching cubes + largest-connected-component + Laplacian smoothing on the GM segmentation volume. This is the expensive step - both the combined mesh (next cell) and the hemisphere-split meshes (further down) are derived from this single computed mesh, rather than each re-running it from scratch.

In [ ]:
def build_cortical_mesh(gm_nii_path):
    """
    Extract a smoothed cortical surface mesh from a GM segmentation NIfTI.

    Returns (mesh, nii) - the trimesh.Trimesh object and the loaded NIfTI image
    (kept around for its affine/shape/voxel-size metadata, saved alongside the
    mesh further down).
    """
    nii = nib.load(str(gm_nii_path))
    data = nii.get_fdata()
    affine = nii.affine  # to map voxels to world space

    # --- Optional: remove cerebellum voxels ---
    # Uncomment and set cbPath to enable. Not used by default - the cerebellum is
    # currently left in the extracted surface.
    # cbPath = dataDriveRoot / 'mri' / 'neurodevelopmentalMRI' / 'Brain' / 'Atlas' / f'ANTS{timePoint}-0Months3T_brain_cerebellum.nii.gz'
    # cbMask = nib.load(str(cbPath)).get_fdata()
    # data[cbMask > 0] = 0

    # zero out edges (if any non-zero voxels are stuck to the boundary)
    data[0, :, :] = 0
    data[-1, :, :] = 0
    data[:, 0, :] = 0
    data[:, -1, :] = 0
    data[:, :, 0] = 0
    data[:, :, -1] = 0

    # pad volume with zeros - prevents edge artifacts
    padded_data = np.pad(data, pad_width=1, mode='constant', constant_values=0)

    # extract surface mesh from padded data
    verts, faces, normals, values = measure.marching_cubes(padded_data, level=0.5, step_size=1)

    # remove padding offset
    verts -= 1

    # Transform voxel coordinates to world space using affine
    vertsWorld = nib.affines.apply_affine(affine, verts)

    # create mesh
    mesh = trimesh.Trimesh(vertices=vertsWorld, faces=faces)

    # keep the largest connected surface component
    components = mesh.split(only_watertight=False)
    mesh = max(components, key=lambda c: c.area)

    # Laplacian smoothing
    filter_laplacian(mesh, lamb=0.1, iterations=5)

    return mesh, nii


mesh, nii = build_cortical_mesh(gmVolPath)

t1Info = {
    'T_ijk2xyz': nii.affine,
    'dims': nii.shape,
    'voxSize': nii.header.get_zooms()
}


## Save the combined mesh, and view it

In [ ]:
# Export for MATLAB/NeuroDOT
sio.savemat(meshesDir / f'corticalSurface{timePoint}mo.mat', {
    'vertices': mesh.vertices,
    'faces': mesh.faces + 1  # convert to 1-based indexing
})
sio.savemat(meshesDir / f't1Info{timePoint}mo.mat', {'t1Info': t1Info})

mesh.show()


## Split into left/right hemisphere meshes

Reuses the mesh already built above (splitting by x-coordinate midline) rather than recomputing marching cubes + smoothing from scratch.

In [ ]:
xCoords = mesh.vertices[:, 0]
midline = np.median(xCoords)

leftFacesIDx = np.all(xCoords[mesh.faces] < midline, axis=1)
rightFacesIDx = np.all(xCoords[mesh.faces] > midline, axis=1)

leftFaces = mesh.faces[leftFacesIDx]
rightFaces = mesh.faces[rightFacesIDx]


def extractSubmesh(faces, allVertices):
    uniqueVertIdx = np.unique(faces)
    remappedFaces = np.searchsorted(uniqueVertIdx, faces)
    subVertices = allVertices[uniqueVertIdx]
    return subVertices, remappedFaces


leftVertices, leftFacesRemapped = extractSubmesh(leftFaces, mesh.vertices)
rightVertices, rightFacesRemapped = extractSubmesh(rightFaces, mesh.vertices)

# Export for MATLAB/NeuroDOT (MATLAB uses 1-based indexing so plus 1)
sio.savemat(meshesDir / f'leftHemisphereMesh{timePoint}mo.mat', {
    'nodes': leftVertices,
    'elements': leftFacesRemapped + 1
})
sio.savemat(meshesDir / f'rightHemisphereMesh{timePoint}mo.mat', {
    'nodes': rightVertices,
    'elements': rightFacesRemapped + 1
})


In [ ]:
# view left hemisphere mesh
leftMesh = trimesh.Trimesh(vertices=leftVertices, faces=leftFacesRemapped)
leftMesh.show()


In [ ]:
# view right hemisphere mesh
rightMesh = trimesh.Trimesh(vertices=rightVertices, faces=rightFacesRemapped)
rightMesh.show()
